In [6]:
# Local environment check - dynamically load local_setup if running locally in VSCode
try:
    dbutils
except NameError:
    import os, sys
    from pathlib import Path
    
    # Walk up to locate project root folder
    curr_path = Path(os.getcwd()).resolve()
    project_root = None
    for _ in range(5):
        if (curr_path / "local_setup.py").exists():
            project_root = curr_path
            break
        curr_path = curr_path.parent
        
    if project_root:
        sys.path.append(str(project_root))
        from local_setup import spark, dbutils, display
    else:
        print("Warning: Could not locate local_setup.py in parent directories!")


In [7]:
import os
import glob
import sys

# Define widgets for dynamic ClientContainer selection
try:
    dbutils.widgets.text("ClientContainer", "new", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "new"

def execute_ddl_scripts(ddl_directory: str, client_container: str):
    sql_files = sorted(glob.glob(os.path.join(ddl_directory, "*.sql")))
    
    if not sql_files:
        print(f"No .sql files found in: {ddl_directory}")
        return

    print(f"Processing {len(sql_files)} SQL file(s) from {ddl_directory}...")
    
    failed_files = []

    for sql_file in sql_files:
        file_name = os.path.basename(sql_file)
        
        try:
            with open(sql_file, 'r', encoding='utf-8') as f:
                sql_content = f.read()
            
            # Replace catalog placeholders with backticks for numerical safety (e.g. `274`)
            sql_content = sql_content.replace("new.silver.", f"`{client_container}`.silver.")
            sql_content = sql_content.replace("new.gold.", f"`{client_container}`.gold.")
            
            # Execute statement in Spark
            spark.sql(sql_content)
            print(f"SUCCESS: {file_name}")
            
        except Exception as e:
            print(f"FAILED: {file_name} | Error: {str(e)}")
            failed_files.append((file_name, str(e)))

    # Final summary check
    if failed_files:
        print(f"\nExecution finished with {len(failed_files)} failure(s):")
        for file, error in failed_files:
            print(f"  - {file}: {error[:100]}")
        raise RuntimeError("DDL batch execution failed.")
    
    print("All DDL scripts executed successfully.")

In [8]:
if __name__ == "__main__":
    # Resolve DDL directory dynamically relative to current notebook workspace location
    import os
    current_dir = os.getcwd()
    DDL_DIR = os.path.abspath(os.path.join(current_dir, "..", "DDL", "DimProvider"))
    execute_ddl_scripts(DDL_DIR, client_container)

Processing 4 SQL file(s) from /home/logidhasan/data/github/claimspan/ClaimsProcessing/DDL/DimProvider...
FAILED: gold_dimprovider.sql | Error: name 'spark' is not defined
FAILED: silver_provider.sql | Error: name 'spark' is not defined
FAILED: silver_provider_hierarchy.sql | Error: name 'spark' is not defined
FAILED: silver_provider_person_bridge.sql | Error: name 'spark' is not defined

Execution finished with 4 failure(s):
  - gold_dimprovider.sql: name 'spark' is not defined
  - silver_provider.sql: name 'spark' is not defined
  - silver_provider_hierarchy.sql: name 'spark' is not defined
  - silver_provider_person_bridge.sql: name 'spark' is not defined


RuntimeError: DDL batch execution failed.